# 03 — Multi-Hop RAG Evaluation
Reuse index from notebook 02. Run multi-hop pipeline on same 200 questions. Compare vs baseline.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

In [ ]:
import json
from config import ACCURACY_DIR, INDEX_DIR

# Load index built in notebook 02
from pipeline.indexer import load_index
index, chunks = load_index(os.path.join(INDEX_DIR, 'baseline_1k'))
print(f'Index loaded: {index.ntotal} vectors, {len(chunks)} chunks')

In [ ]:
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', 'YOUR_KEY_HERE')

from pipeline.generator import GeminiGenerator
from pipeline.embedder import get_model
from multihop.pipeline import run_multihop
from evaluation.metrics import evaluate_dataset

generator   = GeminiGenerator(api_key=GEMINI_API_KEY)
embed_model = get_model()

## Multi-Hop on FinQA

In [ ]:
with open(os.path.join(ACCURACY_DIR, 'finqa_eval_questions.json')) as f:
    finqa_eval = json.load(f)

def multihop_fn(q):
    result = run_multihop(q, index, chunks, embed_model, generator)
    return result['answer']

finqa_mh = evaluate_dataset(
    finqa_eval,
    multihop_fn,
    log_path=os.path.join(ACCURACY_DIR, 'multihop_finqa.jsonl')
)
print('FinQA Multi-Hop:', finqa_mh)

## Multi-Hop on MultiHop-RAG

In [ ]:
with open(os.path.join(ACCURACY_DIR, 'multihop_eval_questions.json')) as f:
    multihop_eval = json.load(f)

multihop_mh = evaluate_dataset(
    multihop_eval,
    multihop_fn,
    log_path=os.path.join(ACCURACY_DIR, 'multihop_multihop.jsonl')
)
print('MultiHop Multi-Hop:', multihop_mh)

## Comparison Table

In [ ]:
import pandas as pd

with open(os.path.join(ACCURACY_DIR, 'baseline_results.json')) as f:
    baseline = json.load(f)

table = pd.DataFrame([
    {'Phase': 'Phase 1 — Baseline',
     'FinQA EM': round(baseline['finqa_em'], 3),
     'MultiHop F1': round(baseline['multihop_f1'], 3),
     'Avg FinQA latency (ms)': round(baseline['finqa_latency_ms'], 0)},
    {'Phase': 'Phase 2 — Multi-Hop',
     'FinQA EM': round(finqa_mh['em'], 3),
     'MultiHop F1': round(multihop_mh['f1'], 3),
     'Avg FinQA latency (ms)': round(finqa_mh['avg_latency_ms'], 0)},
])
print(table.to_string(index=False))

In [ ]:
summary = {
    'finqa_em':    finqa_mh['em'],
    'multihop_f1': multihop_mh['f1'],
    'finqa_latency_ms':    finqa_mh['avg_latency_ms'],
    'multihop_latency_ms': multihop_mh['avg_latency_ms'],
}
with open(os.path.join(ACCURACY_DIR, 'multihop_results.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved multihop_results.json')